<center><h1>Do_Mimi_HW3</h1></center>
<br>
<br>

Name: Mimi Do
<br>
Github Username: mkd222
<br>
USC ID: 8125-6556-10

## 1. Time Series Classification Part 1: Feature Creation/Extraction

### (a) Download Data

#### Package imports

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.stats import bootstrap

#### Get the AReM Data Set

In [2]:
data_path = Path("../data")

print(data_path.exists())

df = pd.read_csv(data_path / "bending1" / "dataset1.csv", comment = "#", header = None)

df.head()

True


,0,1,2,3,4,5,6
0,0,39.25,0.43,22.75,0.43,33.75,1.3
1,250,39.25,0.43,23.00,0.00,33.00,0.0
2,500,39.25,0.43,23.25,0.43,33.00,0.0
3,750,39.50,0.50,23.00,0.71,33.00,0.0
4,1000,39.50,0.50,24.00,0.00,33.00,0.0


#### Checking datasets for anomalies

In [3]:
for folder in data_path.iterdir():
    if folder.is_dir():
        for csv in folder.glob("dataset*.csv"):
            
            try:
                df = pd.read_csv(csv, comment = "#", header = None)

                if df.shape[0] != 480:
                    print(csv, "has", df.shape[0], "rows")

                if df.shape[1] != 7:
                    print(csv, "has", df.shape[1], "columns")

                if df.isnull().sum().sum() > 0:                  # counts and adds total number of missing values
                    print(csv, "has missing values")

            except pd.errors.ParserError:
                print(csv, "has a parsing error")


..\data\bending2\dataset4.csv has 1 columns
..\data\cycling\dataset14.csv has a parsing error
..\data\cycling\dataset9.csv has a parsing error
..\data\sitting\dataset8.csv has 479 rows


-   bending2 dataset4 has 1 column instead of 7
-   cycling dataset14 has parsing error (extra comma at the end, resulting in 8 columns)
-   cycling dataset9 has parsing error (extra comma at the end, resulting in 8 columns)
-   sitting dataset8 has 479 rows instead of 480 excluding comment rows

Initial code without try/except gave a ParseError at row 485. The below cell checks last 5 rows of datasets with ParseError

In [4]:
csv = data_path / "cycling" / "dataset9.csv"

with open(csv) as r:
    lines = r.readlines()

for line in lines[-5:]:
    print(repr(line))

'118750,41.75,1.79,11.00,6.00,16.67,2.49\n'
'119000,36.33,0.47,16.00,3.16,20.33,1.70\n'
'119250,31.50,1.50,21.00,0.00,12.25,7.12\n'
'119500,34.25,6.38,12.67,2.49,15.25,4.21\n'
'119750,38.33,0.94,15.25,2.17,20.33,1.25,'


### (b) Test and Train Data

In [5]:
train_data = []
test_data = []

for folder in data_path.iterdir():
    if folder.is_dir():

        if folder.name in ["bending1", "bending2"]:
            test_csv = ["dataset1.csv", "dataset2.csv"]
        else:
            test_csv = ["dataset1.csv", "dataset2.csv", "dataset3.csv"]

        for csv in folder.glob("dataset*.csv"):
            if csv.name in test_csv:
                test_data.append(csv)
            else:
                train_data.append(csv)

print("Train Data Files:", len(train_data))
print("Test Data Files:", len(test_data))

Train Data Files: 69
Test Data Files: 19


### (c) Feature Extraction

#### i. Research

Minimum, Maximum, Mean, Median, Standard Deviation, Variance, Q1, Q3, IQR, Range, Skewness, RMS

#### ii. Extraction

In [ ]:
all_features = []

all_files = sorted(train_data) + sorted(test_data)

for csv in all_files:
    if csv.parent.name == "bending2" and csv.name == "dataset4.csv":
        df = pd.read_csv(csv, comment = "#", header = None, sep = r"\s+")        # seperate by whitespace

    elif csv.parent.name == "cycling" and csv.name in ["dataset9.csv", "dataset14.csv"]:
        df = pd.read_csv(csv, comment = "#", header = None, usecols = range(7))         # only use first 7 columns, column 8 only has a comma 

    else:
        df = pd.read_csv(csv, comment = "#", header = None)

    features = {}

    for i in range(1, 7):
        time_series = df[i]

        features[f"minimum{i}"] = time_series.min()
        features[f"maximum{i}"] = time_series.max()
        features[f"mean{i}"] = time_series.mean()
        features[f"median{i}"] = time_series.median()
        features[f"standard deviation{i}"] = time_series.std()
        features[f"first quartile{i}"] = time_series.quantile(0.25)
        features[f"third quartile{i}"] = time_series.quantile(0.75)

    all_features.append(features)

df_features = pd.DataFrame(all_features)

df_features.index = range(1, len(df_features) + 1)
df_features.index.name = "Instance"

df_features.head()

,minimum1,maximum1,mean1,median1,standard deviation1,first quartile1,third quartile1,minimum2,maximum2,mean2,...,standard deviation5,first quartile5,third quartile5,minimum6,maximum6,mean6,median6,standard deviation6,first quartile6,third quartile6
Instance,,,,,,,,,,,,,,,,,,,,,
1,35.00,47.40,43.954500,44.33,1.558835,43.00,45.00,0.0,1.70,0.426250,...,1.999604,35.3625,36.50,0.0,1.79,0.493292,0.43,0.513506,0.00,0.94
2,33.00,47.75,42.179812,43.50,3.670666,39.15,45.00,0.0,3.00,0.696042,...,3.849448,30.4575,36.33,0.0,2.18,0.613521,0.50,0.524317,0.00,1.00
3,33.00,45.75,41.678063,41.75,2.243490,41.33,42.75,0.0,2.83,0.535979,...,2.411026,28.4575,31.25,0.0,1.79,0.383292,0.43,0.389164,0.00,0.50
4,37.00,48.00,43.454958,43.25,1.386098,42.50,45.00,0.0,1.58,0.378083,...,2.488862,22.2500,24.00,0.0,5.26,0.679646,0.50,0.622534,0.43,0.87
5,36.25,48.00,43.969125,44.50,1.618364,43.31,44.67,0.0,1.50,0.413125,...,3.318301,20.5000,23.75,0.0,2.96,0.555312,0.49,0.487826,0.00,0.83


#### iii. Standard Deviation

In [7]:
std_features = df_features.std().round(2)

print(std_features)

minimum1               9.57
maximum1               4.39
mean1                  5.34
median1                5.44
standard deviation1    1.77
first quartile1        6.15
third quartile1        5.14
minimum2               0.00
maximum2               5.06
mean2                  1.57
median2                1.41
standard deviation2    0.88
first quartile2        0.95
third quartile2        2.13
minimum3               2.96
maximum3               4.88
mean3                  4.01
median3                4.04
standard deviation3    0.95
first quartile3        4.22
third quartile3        4.17
minimum4               0.00
maximum4               2.18
mean4                  1.17
median4                1.15
standard deviation4    0.46
first quartile4        0.84
third quartile4        1.55
minimum5               6.12
maximum5               5.74
mean5                  5.68
median5                5.81
standard deviation5    1.02
first quartile5        6.10
third quartile5        5.53
minimum6            

In [8]:
def std_sample(data):
    return np.std(data, ddof=1)  # ddof 0 = population , ddof 1 = sample

results = []

for column in df_features.columns:
    values = df_features[column].to_numpy()

    std = std_sample(values)

    confidence_interval = bootstrap(
        (values,),
        std_sample,
        confidence_level = 0.90,
        method = "percentile",     # default n_sample = 9999
        vectorized = False        # process one sample at a time
    )

    results.append({
        "Feature": column,
        "Standard Deviation": std,
        "Lower Confidence Interval": confidence_interval.confidence_interval.low,
        "Upper Confidence Interval": confidence_interval.confidence_interval.high
    })

df_std = pd.DataFrame(results)

display(df_std.round(2))


,Feature,Standard Deviation,Lower Confidence Interval,Upper Confidence Interval
0,minimum1,9.57,8.28,10.76
1,maximum1,4.39,3.33,5.28
2,mean1,5.34,4.71,5.88
3,median1,5.44,4.80,5.99
4,standard deviation1,1.77,1.57,1.95
5,first quartile1,6.15,5.57,6.65
6,third quartile1,5.14,4.32,5.84
7,minimum2,0.00,0.00,0.00
8,maximum2,5.06,4.62,5.41
9,mean2,1.57,1.40,1.70


#### iv. Select Features

Maximum, Mean, Standard Deviation

## 2. ISLR 3.7.4

### (a) Linear Train

Cubic regression would have an equal or lower training RSS than linear regression because there are additional polynomials therefore more flexible

### (b) Linear Test

Linear regression would have a lower test RSS because the true relationship is linear. Cubic regression could overfit test data

### (c) Not Linear Train

Cubic regression would have an equal or lower training RSS than linear regression because there are additional polynomials therefore more flexible

### (d) Not Linear Testing

There is not enough information to tell since the relationship is not linear. It is not specified if the relationship has a weak or strong linear relationship

## 3. ISLR 3.7.3 - Extra Practice 

## 3. ISLR 3.7.5 - Extra Practice 